In [1]:
# --- imports + per-lhid cosmology table (loaded once) ---
import numpy as np
import pandas as pd
import h5py
import os
from os.path import join, exists
from tqdm import tqdm
from colossus.cosmology import cosmology
from colossus.halo import mass_so

COSMO_PARAMS = '/home/x-mho1/git/ltu-cmass-run/params/latin_hypercube_params.txt'
IN_TMPL  = '/anvil/scratch/x-mho1/quijote/Halos/Rockstar/latin_hypercube_HR/{lhid}/out_3_pid.list'
OUT_TMPL = '/anvil/scratch/x-mho1/cmass-ili/quijote/nbodyconc/L1000-N128/{lhid}'
_COSMO_ALL = np.loadtxt(COSMO_PARAMS)  # load params once, index per-lhid

In [2]:
# --- loader: M200c mass, R200c/Rs concentration, parents only, mass cut ---
def load_rockstar_list(path, lhid):
    a = 0.666667                                   # snapshot 3
    z = 1.0 / a - 1.0

    with open(path) as f:                          # read column-name header
        header = f.readline().lstrip('#').strip().split()
    df = pd.read_csv(path, sep=r'\s+', comment='#', header=None,
                     names=header, engine='c')

    df = df[df['PID'] == -1]                        # parents only (drop subhalos)

    pos = df[['X', 'Y', 'Z']].to_numpy(np.float32)    # Mpc/h comoving
    vel = df[['VX', 'VY', 'VZ']].to_numpy(np.float32) # km/s physical
    M200c = df['M200c'].to_numpy(np.float64)          # Msun/h
    Rs = df['Rs'].to_numpy(np.float64)                # kpc/h comoving

    Om0, Ob0, h0, ns_cosmo, sigma8 = _COSMO_ALL[lhid]  # per-lhid cosmology
    cosmology.setCosmology('myCosmo', **{
        'flat': True, 'H0': h0 * 100, 'Om0': Om0, 'Ob0': Ob0,
        'sigma8': sigma8, 'ns': ns_cosmo})

    R200c = mass_so.M_to_R(M200c, z, '200c') * (1.0 + z)  # -> comoving kpc/h
    conc = (R200c / Rs).astype(np.float32)
    with np.errstate(divide='ignore'):
        mass = np.log10(M200c.astype(np.float32))         # log10(Msun/h)

    m = mass >= np.log10(5e12)                            # CHARM mass cut
    return dict(mass=mass[m], pos=pos[m], vel=vel[m], conc=conc[m], a=a)

In [3]:
def save_snapshot(outdir, a, hpos, hvel, hmass, **meta):
    # from rho_to_halo
    with h5py.File(join(outdir, 'halos.h5'), 'a') as f:
        group = f.create_group(f'{a:.6f}')
        group.create_dataset('pos', data=hpos)  # comoving positions [Mpc/h]
        group.create_dataset('vel', data=hvel)  # physical velocities [km/s]
        group.create_dataset('mass', data=hmass)  # halo masses [Msun/h]

        # save other halo metadata (e.g. concentration)
        for key, val in meta.items():
            group.create_dataset(key, data=val)

In [4]:
# --- per-lhid worker: returns a status tag instead of mutating shared state ---
from multiprocessing import Pool

def process_lhid(lhid):
    path, outdir = IN_TMPL.format(lhid=lhid), OUT_TMPL.format(lhid=lhid)

    if exists(join(outdir, 'halos.h5')):   # resumable: skip finished
        return ('skip', lhid, None)
    if not exists(path):                   # missing input catalog
        return ('miss', lhid, path)

    try:
        halos = load_rockstar_list(path, lhid)
        os.makedirs(outdir, exist_ok=True)
        save_snapshot(outdir, a=halos['a'], hpos=halos['pos'],
                      hvel=halos['vel'], hmass=halos['mass'],
                      concentration=halos['conc'])
        return ('done', lhid, None)
    except Exception as e:                 # don't let one bad lhid kill the pool
        return ('err', lhid, f'{type(e).__name__}: {e}')

# --- parallel map over lhids 0..1999 with progress bar ---
N_PROC = 8  # set to your allocated core count

counts = {'done': 0, 'skip': 0, 'miss': 0, 'err': 0}
with Pool(N_PROC) as pool:
    for status, lhid, info in tqdm(
            pool.imap_unordered(process_lhid, range(2000), chunksize=4),
            total=2000, desc='lhids'):
        counts[status] += 1
        if status in ('miss', 'err'):      # surface problems above the bar
            tqdm.write(f'[{lhid}] {status.upper()} {info}')

print(f"done: ok={counts['done']} skip={counts['skip']} "
      f"miss={counts['miss']} err={counts['err']}")

lhids:   0%|          | 0/2000 [00:00<?, ?it/s]

lhids:   4%|▎         | 73/2000 [00:10<04:35,  6.98it/s]<ipython-input-2-29b0b7397743>:24: RuntimeWarning: divide by zero encountered in true_divide
  conc = (R200c / Rs).astype(np.float32)
lhids:   4%|▍         | 85/2000 [00:24<12:28,  2.56it/s]<ipython-input-2-29b0b7397743>:24: RuntimeWarning: divide by zero encountered in true_divide
  conc = (R200c / Rs).astype(np.float32)
lhids:  54%|█████▍    | 1077/2000 [21:22<17:04,  1.11s/it] <ipython-input-2-29b0b7397743>:24: RuntimeWarning: divide by zero encountered in true_divide
  conc = (R200c / Rs).astype(np.float32)
lhids:  61%|██████▏   | 1225/2000 [24:32<21:50,  1.69s/it]<ipython-input-2-29b0b7397743>:24: RuntimeWarning: divide by zero encountered in true_divide
  conc = (R200c / Rs).astype(np.float32)
lhids:  72%|███████▏  | 1437/2000 [28:44<07:35,  1.24it/s]<ipython-input-2-29b0b7397743>:24: RuntimeWarning: divide by zero encountered in true_divide
  conc = (R200c / Rs).astype(np.float32)
lhids:  77%|███████▋  | 1533/2000 [30:41<05

done: ok=1918 skip=82 miss=0 err=0


## Scratch

In [5]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
lhids = rng.choice(2001, size=5, replace=False)  # 0..2000 inclusive

print(f"{'lhid':>6}  {'N_pass':>8}  {'N_sub':>8}  {'%sub':>7}")
tot_pass = tot_sub = 0
for lhid in lhids:
    path = f'/anvil/scratch/x-mho1/quijote/Halos/Rockstar/latin_hypercube_HR/{lhid}/out_3_pid.list'
    with open(path) as f:
        header = f.readline().lstrip('#').strip().split()
    df = pd.read_csv(path, sep=r'\s+', comment='#', header=None, names=header, engine='c')

    # mass cut on M200c (same threshold as loader), applied BEFORE parent filter
    passed = df[df['M200c'] > 5e12]
    n_pass = len(passed)
    n_sub = int((passed['PID'] != -1).sum())
    tot_pass += n_pass
    tot_sub += n_sub
    print(f"{lhid:>6}  {n_pass:>8}  {n_sub:>8}  {100*n_sub/n_pass:>6.2f}%")

print(f"\nOverall: {tot_sub}/{tot_pass} = {100*tot_sub/tot_pass:.2f}% subhalos")

  lhid    N_pass     N_sub     %sub
  1272    946706     64031    6.76%
  1021    636368     39322    6.18%
   539    749619     34735    4.63%
   615    755454     38772    5.13%
  1698    692432     34871    5.04%

Overall: 211731/3780579 = 5.60% subhalos
